In [121]:
import sys
import numpy as np
from scipy.optimize import linprog
import itertools
import os
import time
from functools import partial

In [101]:
import random
import math

In [102]:
random.seed(42)

In [103]:
tests = ['vrp_16_3_1', 'vrp_26_8_1', 'vrp_51_5_1', 'vrp_101_10_1', 'vrp_200_16_1', 'vrp_421_41_1']
thresholds = [(387, 280), (1019, 630), (713, 540), (1193, 830), (3719, 1400), (2392, 2000)]

In [104]:
def load_data(test):
    with open(f"data/{test}") as file:
        lines = file.readlines()
        args = lines[0].split()
        n = int(args[0])
        v = int(args[1])
        c = float(args[2])
        req = list()
        points = list()
        for i in range(n):
            d, x, y = list(map(float, lines[1 + i].split()))
            req.append(d)
            points.append((x, y))
            
        return n, v, c, req, points

In [105]:
def check_vrp(n, v, c, req, points, paths):
    used = [0] * n
    taken = [0] * v

    if len(paths) != v:
        return 1e18
    
    for wh, path in enumerate(paths):
        for i in path:
            taken[wh] += req[i]
            used[i] += 1

    for i in range(v):
        if taken[i] > c:
            return 1e18
            
    for i in range(1, n):
        if used[i] != 1:
            return 1e18
            
    res = 0
    for path in paths:
        if len(path) == 0:
            continue
        res += math.dist(points[0], points[path[0]])
        
        for i in range(len(path) - 1):
            res += math.dist(points[path[i]], points[path[i + 1]])
            
        res += math.dist(points[path[-1]], points[0])

    return res

In [106]:
def passed_cnt(result, idx):
    if result <= thresholds[idx][1]:
        return 2
    elif result <= thresholds[idx][0]:
        return 1
    else:
        return 0

In [107]:
def test_method(method, name, use_file=False):
    print(f"Checking {name}")
    score = 0
    for i, test in enumerate(tests):
        n, v, c, req, points = load_data(test)
        start = time.time()
        
        if not use_file:
            paths = method(n, v, c, req, points)
        else:
            paths = method(test)

        end = time.time()
        elapsed = end - start
        print(f"Execution time: {elapsed:.4f} seconds")
            
        result = check_vrp(n, v, c, req, points, paths)
        passed = passed_cnt(result, i)
        
        if passed == 1:
            score += 3
        elif passed == 2:
            score += 5

        print(f"Target function {test}: {result}")
        print(f"Passed {test}: {passed}")

    print(f"Score: {score}")

Пока будем решать задачу в декомпозированном виде:
1. Найдем сопоставление каждого покупателя -- доставщику
2. Решим на доставщике TSP

Идейно будем решать задачу именно так, но я не хочу при каждом изменении сопоставления заново честно оптимизировать TSP. Поэтому воспользуемся простой эвристикой: если цикл меняется незначительно — например, мы добавляем или удаляем одну точку, — то и оптимальный цикл, скорее всего, изменится не слишком сильно.

А именно: при добавлении новой точки я буду искать для неё оптимальную вставку в текущий цикл, а при удалении точки оставлять порядок остальных точек без изменений.

В качестве жадного решения будем идею похожую на эвристику Кларка-Райта ([Clarke-Wright savings algorithm](https://medium.com/@ranjitodedra/clarke-wright-algorithm-for-ev-routing-research-2771473bcab8)).

Изначально каждая вершина образует отдельный цикл вида `0 -> v -> 0`.
Далее на каждом шаге будем искать два цикла, которые можно допустимо склеить в один. Среди всех
возможных склеек выбираем ту, которая даёт наибольшее улучшение.

Есть ли есть улучшающая склейка (уменьшающая таргет), то мы ее всегда делаем. Иначе все склейки ухудшают счет и мы их делаем, пока машин больше `V`.

In [108]:
!g++ -O2 -std=c++2a cpp_methods/greedy.cpp -o tmp/greedy

In [109]:
def greedy_vrp(test_file): 
    os.system(f"./tmp/greedy data/{test_file}")
    with open("tmp/ans.txt") as file:
        lines = file.readlines()
        decomp = list()
        for i in range(len(lines)):
            decomp.append(list(map(int, lines[i].split())))
        return decomp

In [110]:
test_method(greedy_vrp, "greedy_vrp", True)

Checking greedy_vrp
Execution time: 0.1929 seconds
Target function vrp_16_3_1: 1e+18
Passed vrp_16_3_1: 0
Execution time: 0.0113 seconds
Target function vrp_26_8_1: 1e+18
Passed vrp_26_8_1: 0
Execution time: 0.0097 seconds
Target function vrp_51_5_1: 1e+18
Passed vrp_51_5_1: 0
Execution time: 0.0118 seconds
Target function vrp_101_10_1: 833.5085732089909
Passed vrp_101_10_1: 1
Execution time: 0.0169 seconds
Target function vrp_200_16_1: 1e+18
Passed vrp_200_16_1: 0
Execution time: 0.0623 seconds
Target function vrp_421_41_1: 1958.7462601014645
Passed vrp_421_41_1: 2
Score: 8


Интересно...

Как мы видим, жадное решение получило отличный скор на последнем тесте, однако на некоторых тестах оно выдаёт невалидные решения: алгоритм не может склеить циклы достаточное количество раз из-за превышения $\text{capacity}$ курьера.

Попробуем добавить локальную оптимизацию: будем брать вершину из одного цикла и пытаться перенести её в другой цикл. Чтобы бороться с невалидными циклами, добавим к стоимости решения штраф за превышение $\text{capacity}$, умноженный на $\lambda$.

Так как мы ввели понятие штрафа, немного улучшим и жадное решение. А именно: если жадный алгоритм не нашёл feasible-решение, он продолжает решать ту же задачу, но теперь циклы можно склеивать даже при превышении $\text{capacity}$, добавляя при этом тот же штраф.

In [111]:
!g++ -O2 -std=c++2a cpp_methods/local_opt.cpp -o tmp/local_opt

In [112]:
def local_opt_vrp(test_file): 
    os.system(f"./tmp/local_opt data/{test_file}")
    with open("tmp/ans.txt") as file:
        lines = file.readlines()
        decomp = list()
        for i in range(len(lines)):
            decomp.append(list(map(int, lines[i].split())))
        return decomp

In [113]:
test_method(local_opt_vrp, "local_opt_vrp", True)

Checking local_opt_vrp
Execution time: 0.1825 seconds
Target function vrp_16_3_1: 290.4187167384623
Passed vrp_16_3_1: 1
Execution time: 0.0140 seconds
Target function vrp_26_8_1: 1e+18
Passed vrp_26_8_1: 0
Execution time: 0.0117 seconds
Target function vrp_51_5_1: 585.2635459339223
Passed vrp_51_5_1: 1
Execution time: 0.0128 seconds
Target function vrp_101_10_1: 827.2746190907471
Passed vrp_101_10_1: 2
Execution time: 0.0298 seconds
Target function vrp_200_16_1: 1e+18
Passed vrp_200_16_1: 0
Execution time: 0.1649 seconds
Target function vrp_421_41_1: 1953.879818891265
Passed vrp_421_41_1: 2
Score: 16


Заимпрувили!

А теперь добавим локальную оптимизацию, где мы меняем местами две вершины из разных циклов. Применяем ее, если не нашлась операция, описанная ранее.

In [114]:
!g++ -O2 -std=c++2a cpp_methods/tuned_local_opt.cpp -o tmp/tuned_local_opt

In [115]:
def tuned_local_opt_vrp(test_file): 
    os.system(f"./tmp/tuned_local_opt data/{test_file}")
    with open("tmp/ans.txt") as file:
        lines = file.readlines()
        decomp = list()
        for i in range(len(lines)):
            decomp.append(list(map(int, lines[i].split())))
        return decomp

In [116]:
test_method(tuned_local_opt_vrp, "tuned_local_opt", True)

Checking tuned_local_opt
Execution time: 0.1887 seconds
Target function vrp_16_3_1: 285.9566139053619
Passed vrp_16_3_1: 1
Execution time: 0.0149 seconds
Target function vrp_26_8_1: 607.6509456345309
Passed vrp_26_8_1: 2
Execution time: 0.0135 seconds
Target function vrp_51_5_1: 564.3255458794799
Passed vrp_51_5_1: 1
Execution time: 0.0144 seconds
Target function vrp_101_10_1: 825.5411704596885
Passed vrp_101_10_1: 2
Execution time: 0.0531 seconds
Target function vrp_200_16_1: 1458.1919573093974
Passed vrp_200_16_1: 1
Execution time: 0.3736 seconds
Target function vrp_421_41_1: 1945.3088284221412
Passed vrp_421_41_1: 2
Score: 24


УФФФФ

Теперь добавим стохастику: будем случайным образом разбивать циклы на более короткие. Для этого будем последовательно идти по циклу и с вероятностью $p$ разрывать его в текущей вершине. Все вершины, накопленные с момента последнего разрыва, будут образовывать новый цикл.

К полученным новым циклам будем применять жадный алгоритм, описанный в начале, а затем запускать локальные оптимизации.

In [140]:
!g++ -O2 -std=c++2a cpp_methods/stoch_local_opt.cpp -o tmp/stoch_local_opt

In [141]:
def stoch_local_opt_vrp(test_file, prob): 
    os.system(f"./tmp/stoch_local_opt data/{test_file} {prob}")
    with open("tmp/ans.txt") as file:
        lines = file.readlines()
        decomp = list()
        for i in range(len(lines)):
            decomp.append(list(map(int, lines[i].split())))
        return decomp

In [142]:
test_method(partial(stoch_local_opt_vrp, prob=0.01), "stoch_local_opt", True)

Checking stoch_local_opt
Execution time: 6.4397 seconds
Target function vrp_16_3_1: 278.9849406063866
Passed vrp_16_3_1: 2
Execution time: 6.0042 seconds
Target function vrp_26_8_1: 607.6509456345309
Passed vrp_26_8_1: 2
Execution time: 6.0039 seconds
Target function vrp_51_5_1: 554.5667962628403
Passed vrp_51_5_1: 1
Execution time: 6.0055 seconds
Target function vrp_101_10_1: 825.5411704596887
Passed vrp_101_10_1: 2
Execution time: 6.0057 seconds
Target function vrp_200_16_1: 1425.8094530941573
Passed vrp_200_16_1: 1
Execution time: 6.0054 seconds
Target function vrp_421_41_1: 1945.3088284221415
Passed vrp_421_41_1: 2
Score: 26


In [143]:
test_method(partial(stoch_local_opt_vrp, prob=0.1), "stoch_local_opt", True)

Checking stoch_local_opt
Execution time: 6.0041 seconds
Target function vrp_16_3_1: 278.9849406063866
Passed vrp_16_3_1: 2
Execution time: 6.0041 seconds
Target function vrp_26_8_1: 607.6509456345308
Passed vrp_26_8_1: 2
Execution time: 6.0044 seconds
Target function vrp_51_5_1: 527.6748210713286
Passed vrp_51_5_1: 2
Execution time: 6.0053 seconds
Target function vrp_101_10_1: 825.5411704596887
Passed vrp_101_10_1: 2
Execution time: 6.0055 seconds
Target function vrp_200_16_1: 1376.577291874546
Passed vrp_200_16_1: 2
Execution time: 6.0053 seconds
Target function vrp_421_41_1: 1936.9424419350128
Passed vrp_421_41_1: 2
Score: 30


In [144]:
test_method(partial(stoch_local_opt_vrp, prob=0.25), "stoch_local_opt", True)

Checking stoch_local_opt
Execution time: 6.0040 seconds
Target function vrp_16_3_1: 278.98494060638666
Passed vrp_16_3_1: 2
Execution time: 6.0052 seconds
Target function vrp_26_8_1: 607.6509456345308
Passed vrp_26_8_1: 2
Execution time: 6.0055 seconds
Target function vrp_51_5_1: 527.6748210713288
Passed vrp_51_5_1: 2
Execution time: 6.0052 seconds
Target function vrp_101_10_1: 825.5411704596885
Passed vrp_101_10_1: 2
Execution time: 6.0051 seconds
Target function vrp_200_16_1: 1361.9651973180391
Passed vrp_200_16_1: 2
Execution time: 6.0048 seconds
Target function vrp_421_41_1: 1927.1247328462082
Passed vrp_421_41_1: 2
Score: 30


In [145]:
test_method(partial(stoch_local_opt_vrp, prob=0.3), "stoch_local_opt", True)

Checking stoch_local_opt
Execution time: 6.0041 seconds
Target function vrp_16_3_1: 279.6017425625562
Passed vrp_16_3_1: 2
Execution time: 6.0045 seconds
Target function vrp_26_8_1: 607.6509456345308
Passed vrp_26_8_1: 2
Execution time: 6.0051 seconds
Target function vrp_51_5_1: 527.6748210713287
Passed vrp_51_5_1: 2
Execution time: 6.0049 seconds
Target function vrp_101_10_1: 825.5411704596886
Passed vrp_101_10_1: 2
Execution time: 6.0196 seconds
Target function vrp_200_16_1: 1386.3128046730997
Passed vrp_200_16_1: 2
Execution time: 6.0052 seconds
Target function vrp_421_41_1: 1924.040170670808
Passed vrp_421_41_1: 2
Score: 30


In [146]:
test_method(partial(stoch_local_opt_vrp, prob=0.5), "stoch_local_opt", True)

Checking stoch_local_opt
Execution time: 6.0043 seconds
Target function vrp_16_3_1: 279.6017425625562
Passed vrp_16_3_1: 2
Execution time: 6.0051 seconds
Target function vrp_26_8_1: 607.6509456345309
Passed vrp_26_8_1: 2
Execution time: 6.0053 seconds
Target function vrp_51_5_1: 527.6748210713288
Passed vrp_51_5_1: 2
Execution time: 6.0046 seconds
Target function vrp_101_10_1: 825.5411704596889
Passed vrp_101_10_1: 2
Execution time: 6.0053 seconds
Target function vrp_200_16_1: 1399.223666762031
Passed vrp_200_16_1: 2
Execution time: 6.0049 seconds
Target function vrp_421_41_1: 1943.184409618884
Passed vrp_421_41_1: 2
Score: 30


Круто! Итоговая вундервафля показала себя мощно. Метрики при $p \approx 0.1-0.3$ примерно одинаково хороши.

Хотя мы и прошли все тесты, я написал еще оптимальное решение, которое работает за $\mathcal{O}(V \cdot 3^N)$. А также находит оптимальный порог в 1 тесте.

1. Сначала считаем динамику, аналогичную TSP:

   `dp[mask][last]` — минимальная длина пути, который начинается в `0`, посещает всех клиентов из `mask` и заканчивается в клиенте `last`.

   База:

   `dp[1 << i][i] = dist(0, i)`

   Переход:

   `dp[mask | (1 << nxt)][nxt] = min(dp[mask | (1 << nxt)][nxt], dp[mask][last] + dist(last, nxt))`

   Дальше для каждой маски считаем лучший цикл одной машины. Маски, в которых суммарный спрос клиентов превосходит `cap`, не учитываем:

   `opt_val[mask] = min(dp[mask][last] + dist(last, 0))`

   Всё это можно сделать за $\mathcal{O}(N^2 \cdot 2^N)$.

2. Далее считаем динамику по числу использованных машин:

   `covered[k][mask]` — минимальная стоимость покрыть множество клиентов `mask`, используя `k` машин.

   База:

   `covered[0][0] = 0`

   Переход: перебираем подмаску `s` множества `mask`, которую будет обслуживать последняя машина:

   `covered[k + 1][mask] = min(covered[k + 1][mask], covered[k][mask ^ s] + opt_val[s])`

   Асимптотика этого шага $\mathcal{O}(V \cdot 3^N)$, и это самая тяжёлая часть алгоритма.

In [151]:
!g++ -O2 -std=c++2a cpp_methods/dp.cpp -o tmp/dp

In [152]:
def dp_vrp(test_file): 
    os.system(f"./tmp/dp data/{test_file}")
    with open("tmp/ans.txt") as file:
        lines = file.readlines()
        decomp = list()
        for i in range(len(lines)):
            decomp.append(list(map(int, lines[i].split())))
        return decomp

In [153]:
test_method(dp_vrp, "dp_vrp", True)

Checking dp_vrp
Execution time: 0.6991 seconds
Target function vrp_16_3_1: 278.7262997478473
Passed vrp_16_3_1: 2
Execution time: 0.0038 seconds
Target function vrp_26_8_1: 1e+18
Passed vrp_26_8_1: 0
Execution time: 0.0034 seconds
Target function vrp_51_5_1: 1e+18
Passed vrp_51_5_1: 0
Execution time: 0.0033 seconds
Target function vrp_101_10_1: 1e+18
Passed vrp_101_10_1: 0
Execution time: 0.0034 seconds
Target function vrp_200_16_1: 1e+18
Passed vrp_200_16_1: 0
Execution time: 0.0040 seconds
Target function vrp_421_41_1: 1e+18
Passed vrp_421_41_1: 0
Score: 5


Данный метод не очень интересно запускать на больших тестах, поэтому просто посмотрим его значения на тестах, где $N \leq 21$.

In [157]:
small_files = ["vrp_5_4_1", "vrp_16_3_1", "vrp_16_5_1", "vrp_21_4_1", "vrp_21_4_1", "vrp_21_6_1"]

In [158]:
for file in small_files:
    decomp = dp_vrp(file)
    n, v, c, req, points = load_data(file)
    value = check_vrp(n, v, c, req, points, decomp)
    print(file, value)

vrp_5_4_1 68.2842712474619
vrp_16_3_1 278.7262997478473
vrp_16_5_1 334.9638862376141
vrp_21_4_1 358.40227415039607
vrp_21_4_1 358.40227415039607
vrp_21_6_1 430.88467542422376


Итог:
1. Написал точное решение через динамику по подмножествам, которое работает за $\mathcal{O}(V \cdot 3^N)$.
2. Реализовал жадник, похожий на эвристику Кларка—Райта. Он пробил пару тестов.
3. Добавили переносы вершин между циклами, штраф за превышение capacity, чтобы не отличать невалидные решения и плохие по метрикам. Совокупность данных эвристик уже пробило много тестов.
4. Добавление локальной оптимизации swap'a вершин еще сильно улучшило результаты.
5. Финальной эвристикой служит добавление стохастики: решение случайно разбивается, затем снова применяется жадная склейка и локальный поиск. Это даёт вариант LNS, который проходит все пороги.
